# LC 150 — Evaluate Reverse Polish Notation
**Difficulty:** Medium | **Category:** Stack | **Pattern:** Expression Evaluation

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> In RPN every operator immediately
follows its two operands. Use a stack of integers — push numbers,
and on each operator pop two values, compute, and push the result.
Division must truncate toward zero: use int(a/b), not a//b.
</div>

## Official Problem Statement

You are given an array of strings `tokens` that represents an
arithmetic expression in Reverse Polish Notation.

Evaluate the expression. Return an integer that represents the
value of the expression.

**Note:**
- The valid operators are `'+'`, `'-'`, `'*'`, and `'/'`.
- Each operand may be an integer or another expression.
- Division between two integers always truncates toward zero.
- There will not be any division by zero.
- The input represents a valid arithmetic expression in RPN.
- The answer and all intermediate calculations fit in a 32-bit
  integer.

**Constraints:**
- `1 <= tokens.length <= 10^4`
- `tokens[i]` is either an operator or an integer in
  `[-200, 200]`

## What This Is Actually Asking

Reverse Polish Notation (RPN) is a way to write math expressions
without parentheses — the operator comes after its two operands.

So `3 4 +` means `3 + 4 = 7`, and `5 1 2 + 4 * + 3 -` works out
step by step from left to right.

You just need to scan the token list once, using a stack to hold
intermediate results until an operator consumes them.

The tricky detail: division must truncate toward zero, which
Python's `//` operator does NOT do for negative numbers.

## Walk Through an Example by Hand

Input: `tokens = ["2","1","+","3","*"]`
Expected: `9`  (because (2+1)*3 = 9)

```
token  action              stack
-----  ------------------  -----------
"2"    push 2              [2]
"1"    push 1              [2, 1]
"+"    pop 1,2; push 2+1  [3]
"3"    push 3              [3, 3]
"*"    pop 3,3; push 3*3   [9]

Return stack[0] = 9
```

Note the pop order: `b = stack.pop()` then `a = stack.pop()`.
For `a - b` and `a / b` the ORDER matters — b was pushed last.

## The Picture

Stack evolution for `["5","1","2","+","4","*","+","3","-"]`
Expected: 14

```
token  stack after token
-----  ---------------------------
"5"    [5]
"1"    [5, 1]
"2"    [5, 1, 2]
"+"    pop 2,1 → 1+2=3       [5, 3]
"4"    [5, 3, 4]
"*"    pop 4,3 → 3*4=12       [5, 12]
"+"    pop 12,5 → 5+12=17    [17]
"3"    [17, 3]
"-"    pop 3,17 → 17-3=14    [14]

Return 14  ✓

Pop order diagram for operator '-':

  before pop:       step 1: b=pop()   step 2: a=pop()
  +------+
  |  3   | <- top   b = 3             a = 17
  +------+
  |  17  |          result = a - b = 17 - 3 = 14
  +------+
```

## When To Use This Pattern

- When you see **postfix notation** or tokens that combine numbers
  and operators, think **stack**.
- When you need to **defer an operation** until you have all
  operands, think **stack**.
- When results of sub-expressions become **inputs to later
  operators**, think **stack**.
- When the problem involves **expression evaluation** of any kind
  (infix, postfix, prefix), think **stack**.
- When order of operands matters (subtraction, division),
  remember: **pop b first, then a**.

## The Approach

Iterate through each token in the list. If it is a number
(could be negative, so check if it's not one of the four
operators), convert to int and push onto the stack.

If the token is an operator, pop b then pop a (order matters for
subtraction and division), apply the operation, and push the
integer result back.

For division, use `int(a / b)` rather than `a // b` to correctly
truncate toward zero for negative results (e.g., -7 / 2 = -3,
not -4).

After all tokens are processed, the single remaining value on
the stack is the answer.

In [3]:
from typing import List  # type hint for token list

In [4]:
def test_harness(func):
    """
    Runs test cases for LC 150 Evaluate Reverse Polish Notation.
    Each case: (tokens_list, expected_int_result)
    """
    cases = [
        # (tokens,                            expected)
        (["2","1","+","3","*"],              9),
        (["4","13","5","/","+"],             6),
        (["10","6","9","3","+","-11","*",
          "/","*","17","+","5","+"],         22),
        (["3","4","+"],                       7),   # simple add
        (["5","3","-"],                       2),   # subtraction order
        (["6","2","/"],                       3),   # integer division
        (["-7","2","/"],                     -3),  # truncate toward zero
        (["7","-2","/"],                     -3),  # truncate toward zero
        (["3"],                               3),   # single number
        (["2","3","*","4","*"],             24),   # chain multiply
    ]
    passed = 0
    for tokens, expected in cases:
        result = func(tokens)
        status = "PASSED" if result == expected else "FAILED"
        if status == "FAILED":
            print(
                f"  {status} | tokens={tokens} "
                f"expected={expected} got={result}"
            )
        else:
            passed += 1
    total = len(cases)
    print(f"\nResult: {passed}/{total} tests passed.")

In [21]:
def eval_rpn(tokens: List[str]) -> int:
    """
    Evaluate an arithmetic expression in Reverse Polish Notation.

    Strategy:
      - Maintain a stack of integers.
      - For each token: if not an operator push int(token).
      - On operator: pop b, pop a, compute, push result.
      - Division: int(a/b) to truncate toward zero.
      - Return stack[0] at the end.

    Args:
        tokens: list of string tokens in RPN order
    Returns:
        integer result of the expression
    """
    stack = []
    ops = {'+', '-', '*', '/'}  # set, not string
    for tk in tokens:
        if tk not in ops:
            stack.append(int(tk))
        elif tk == "+":
            stack.append( stack.pop() + stack.pop() )
        elif tk == "-":
            num =  stack.pop()
            stack.append( stack.pop() - num )
        elif tk == "*":
            stack.append( stack.pop() * stack.pop() )
        elif tk == "/":
            num = stack.pop()
            stack.append( int(stack.pop() /num))
    return(stack[0])

# --- Debug prints (remove before submitting) ---
"""
1
9
-3
2
3
22

Result: 10/10 tests passed.
"""
# Simple addition — expect 9
print(eval_rpn(["2","1","-"]))       # 1
                   
# Simple addition — expect 9
print(eval_rpn(["2","1","+","3","*"]))       # 9

# Division with truncation — expect -3
print(eval_rpn(["-7","2","/"]))              # -3
# Subtraction order check — expect 2 (5-3 not 3-5)
print(eval_rpn(["5","3","-"]))               # 2

# Single number — expect 3
print(eval_rpn(["3"]))                       # 3

# Full example — expect 22
print(eval_rpn(
    ["10","6","9","3","+","-11","*",
     "/","*","17","+","5","+"]
))    # 22          
test_harness(eval_rpn)

1
9
-3
2
3
22

Result: 10/10 tests passed.


In [ ]:

# Division with truncation — expect -3
print(eval_rpn(["-7","2","/"]))              # -3

# Subtraction order check — expect 2 (5-3 not 3-5)
print(eval_rpn(["5","3","-"]))               # 2

# Single number — expect 3
print(eval_rpn(["3"]))                       # 3

# Full example — expect 22
print(eval_rpn(
    ["10","6","9","3","+","-11","*",
     "/","*","17","+","5","+"]
))                        

In [ ]:
# Uncomment and run when solution is ready
# test_harness(eval_rpn)

## Complexity

| Approach        | Time   | Space  |
|-----------------|--------|--------|
| Brute force (recursive eval) | O(n²) | O(n) |
| Optimal (stack) | O(n)   | O(n)   |

- **Time O(n):** each token is processed exactly once.
- **Space O(n):** in the worst case (all numbers, one operator
  at end) the stack holds n-1 values.

## Real World Connection

In AWS Glue and Spark, expression trees for computed columns are
internally represented in a postfix-like form. The execution
engine walks the expression using a stack to evaluate nested
transformations — exactly the RPN pattern.

At Citi, risk calculation engines evaluate complex pricing
formulas serialized as token streams across services. Using
RPN evaluation avoids the overhead of building a full parse tree
for each calculation.

In data engineering, formula columns in spreadsheet-style
transformation tools (like Excel's engine or dbt macros) compile
user formulas into postfix notation for fast stack-based
evaluation during batch processing.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra

In [19]:
-7 // 2 

-4

In [20]:
int(-7/2)

-3